In [13]:
import torch
import torchvision
import torch.utils
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.models import vgg19, VGG19_Weights
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import Subset,DataLoader
from torch.utils.data import Dataset
from torchvision.utils import make_grid, save_image
import os

In [14]:
latent_dim = 100    # Dimensionality of the latent z vector
img_channels = 3    # Number of image channels (1 for grayscale, 3 for color)
feature_map_G = 64  # Base number of feature maps in Generator
feature_map_D = 64 

In [15]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1, feature_map_base=64):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        self.init_size = 4  # initial spatial size (we'll upsample from 4x4)
        # Project and reshape layer: latent_dim to feature_map_base*8 feature maps of size 4x4
        self.fc = nn.Linear(latent_dim, feature_map_base*8 * self.init_size * self.init_size)
        # Transposed convolutional layers to scale up to 64x64
        # Each ConvTranspose2d layer halves the feature map depth and doubles spatial size (until 64x64)
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(feature_map_base*8),
            nn.Upsample(scale_factor=2),  # 4x4 -> 8x8 (alternative to ConvTranspose for simplicity)
            nn.Conv2d(feature_map_base*8, feature_map_base*4, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(feature_map_base*4),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),  # 8x8 -> 16x16
            nn.Conv2d(feature_map_base*4, feature_map_base*2, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(feature_map_base*2),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),  # 16x16 -> 32x32
            nn.Conv2d(feature_map_base*2, feature_map_base, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(feature_map_base),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2),  # 32x32 -> 64x64
            nn.Conv2d(feature_map_base, img_channels, kernel_size=3, stride=1, padding=1),
            nn.Tanh()  # output pixels in range [-1, 1]
        )
        
    def forward(self, z):
        # FC to start: project latent vector to a 4x4 feature map block
        out = self.fc(z)
        out = out.view(out.size(0), -1, self.init_size, self.init_size)  # reshape to (batch, feature_map_base*8, 4, 4)
        img = self.conv_blocks(out)  # upsample through conv blocks to 64x64
        return img

In [16]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=1, feature_map_base=64):
        super(Discriminator, self).__init__()
        # Convolutional downsampling: 64x64 -> 1x1
        # Note: No batch norm in first layer per DCGAN convention for stability.
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(img_channels, feature_map_base, kernel_size=4, stride=2, padding=1),   # 64 -> 32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base, feature_map_base*2, kernel_size=4, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(feature_map_base*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base*2, feature_map_base*4, kernel_size=4, stride=2, padding=1),  # 16 -> 8
            nn.BatchNorm2d(feature_map_base*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base*4, feature_map_base*8, kernel_size=4, stride=2, padding=1),  # 8 -> 4
            nn.BatchNorm2d(feature_map_base*8),
            nn.LeakyReLU(0.2, inplace=True)
            # At this point, feature_map_base*8 feature maps of size 4x4
        )
        # Final output layer: map the 4x4x(feature_map_base*8) features to a single scalar score
        # We use a linear layer after flattening, rather than a conv, for clarity.
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(feature_map_base*8 * 4 * 4, 1)
        # Note: No sigmoid here because we'll use WGAN-GP loss which expects raw scores.

    def forward(self, img):
        features = self.conv_blocks(img)
        flat = self.flatten(features)
        score = self.fc(flat)
        return score

In [17]:
class Encoder(nn.Module):
    def __init__(self, img_channels=1, latent_dim=100, feature_map_base=64):
        super(Encoder, self).__init__()
        # The Encoder will mirror the Discriminator's conv structure (but we include BN in first layer here).
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(img_channels, feature_map_base, kernel_size=4, stride=2, padding=1),   # 64 -> 32
            nn.BatchNorm2d(feature_map_base),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base, feature_map_base*2, kernel_size=4, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(feature_map_base*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base*2, feature_map_base*4, kernel_size=4, stride=2, padding=1),  # 16 -> 8
            nn.BatchNorm2d(feature_map_base*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_map_base*4, feature_map_base*8, kernel_size=4, stride=2, padding=1),  # 8 -> 4
            nn.BatchNorm2d(feature_map_base*8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # After conv_blocks, we have feature_map_base*8 feature maps of size 4x4. Flatten and use a linear layer to get latent vector.
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(feature_map_base*8 * 4 * 4, latent_dim)
    
    def forward(self, img):
        x = self.conv_blocks(img)
        flat = self.flatten(x)
        z = self.fc(flat)              # latent vector (not constrained by an activation here; can be any real values)
        return z

In [18]:
G = Generator(latent_dim, img_channels, feature_map_G)
D = Discriminator(img_channels, feature_map_D)
E = Encoder(img_channels, latent_dim, feature_map_D)

In [19]:
optimizer_G = optim.Adam(G.parameters(), lr=1e-4, betas=(0.5, 0.9))
optimizer_D = optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.9))
lambda_gp = 10.0  # Gradient penalty coefficient
num_epochs_gan=30

In [20]:
for epoch in range(num_epochs_gan):
    for i, real_imgs in enumerate(train_loader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        # ========== Train Discriminator D ========== 
        optimizer_D.zero_grad()
        # Sample random latent vectors
        z = torch.randn(batch_size, latent_dim, device=device)
        fake_imgs = G(z).detach()   # generate fake images; detach G so gradients don't flow to G here
        
        # Discriminator outputs
        real_scores = D(real_imgs)      # D(x)
        fake_scores = D(fake_imgs)      # D(G(z))
        
        # Gradient penalty: interpolate between real and fake
        alpha = torch.rand(batch_size, 1, 1, 1, device=device)
        # x_hat = alpha * real + (1 - alpha) * fake
        interpolated = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)
        interp_scores = D(interpolated)
        # Compute gradients of D(interpolated) wrt interpolated data
        grad_outputs = torch.ones_like(interp_scores, device=device)
        gradients = torch.autograd.grad(
            outputs=interp_scores, inputs=interpolated,
            grad_outputs=grad_outputs,
            create_graph=True, retain_graph=True, only_inputs=True
        )[0]
        # Calculate gradient norm
        gradients = gradients.view(batch_size, -1)
        grad_norm = torch.linalg.norm(gradients, dim=1)
        # Gradient penalty loss
        gp_loss = lambda_gp * ((grad_norm - 1) ** 2).mean()
        
        # WGAN-GP discriminator loss
        d_loss = - real_scores.mean() + fake_scores.mean() + gp_loss
        
        # Backprop and update D
        d_loss.backward()
        optimizer_D.step()
        
        # Optional: multiple D updates could be done here for WGAN's n_critic
        
        # ========== Train Generator G (after n_critic D updates) ========== 
        if i % n_critic == 0:
            optimizer_G.zero_grad()
            # Generate new fake images for generator update
            z = torch.randn(batch_size, latent_dim, device=device)
            gen_imgs = G(z)
            gen_scores = D(gen_imgs)
            # WGAN generator loss (maximize D's score for generated images)
            g_loss = - gen_scores.mean()
            # Backprop and update G
            g_loss.backward()
            optimizer_G.step()
        
    print(f"Epoch {epoch}: d_loss={d_loss.item():.3f}, g_loss={g_loss.item():.3f}")

NameError: name 'train_loader' is not defined